In [ ]:
%load_ext autoreload
%autoreload 3
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import sys, os, pickle, warnings
import numpy as np
import pandas as pd

sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/spe-1/spe1_helper_modules/')
from config import (SPE1_PICKLE_ROOT, DICT_CELL_TYPE, DICT_PATCH_TYPE,
                    DICT_CORT_DEPTH, DICT_DARK_NEURONS, DICT_EAP_WAV)
from pop_ridge_utils import load_population_results, aggregate_population, run_r2_tests
from pop_metadata_utils import (
    build_metadata_df,
    plot_cell_sig_heatmap,
    plot_r2_by_group,
    plot_alpha_by_group,
    plot_metadata_vs_mean_r2,
    plot_n_spikes_confound,
    plot_duration_rate_vs_metadata,
    run_beta_metadata_analysis,
    plot_predictor_set_comparison,
)
from ridge_regression_utils import FEAT_LABELS

warnings.filterwarnings('ignore')

# Population metadata analysis — spe-1

Characterise which cells and which cell properties drive the population-level
waveform → LFP predictability found in `pop_ridge_regression.ipynb`.

**Questions addressed**

| § | Question |
|---|---|
| 1 | Which cells have significant models, and for which targets? |
| 2 | Do CV R² and regularisation strength vary across LFP target groups? |
| 3 | Does cell identity (type, depth, dark-neuron) predict mean R²? |
| 4 | Is n_spikes a confound — driven by firing rate or recording length? |
| 5 | Are recording duration and firing rate unevenly distributed across cell metadata groups? |
| 6 | For R²-significant targets, does cell metadata explain *directionality* of beta weights? |
| 7 | Does adding log-ISI to waveform features improve prediction? |

In [ ]:
# ── Load ridge regression results ─────────────────────────────────────────────
RIDGE_PICKLE_DIR = os.path.join(SPE1_PICKLE_ROOT, 'ridge_regression_pickles')

all_results, cell_ids, target_names, predictor_sets = load_population_results(RIDGE_PICKLE_DIR)

feat_labels   = FEAT_LABELS
target_labels = (
    [f'Pre {l}'     for l in feat_labels] +
    [f'Pre−BL {l}'  for l in feat_labels] +
    [f'Post {l}'    for l in feat_labels] +
    [f'Post−BL {l}' for l in feat_labels] +
    [f'Δ {l}'       for l in feat_labels]
)

# Load population summary pickle
pop_path = os.path.join(RIDGE_PICKLE_DIR, 'population_ridge_results.pkl')
with open(pop_path, 'rb') as f:
    pop = pickle.load(f)

r2_pop        = pop['r2_pop']
sig_pop       = pop['sig_pop']
beta_pop      = pop['beta_pop']
df_tests      = pop['df_tests']
mean_beta_mat = pop['mean_beta_mat']
sig_beta_mat  = pop['sig_beta_mat']

# Load or recompute df_r2 (Wilcoxon median R²>0 population test)
df_r2 = pop.get('df_r2', None)
if df_r2 is None:
    print('df_r2 not in pickle — recomputing …')
    r2_tmp, sig_tmp, _ = aggregate_population(all_results, cell_ids, target_names, predictor_sets)
    df_r2 = run_r2_tests(r2_tmp, target_names, target_labels, predictor_sets)
    print(f'  → {df_r2.sig_r2.sum()} significant targets')
else:
    print(f'Loaded df_r2: {len(df_r2)} rows  |  {df_r2.sig_r2.sum()} significant')

print(f'\n{len(cell_ids)} cells  |  {len(target_names)} targets')

In [ ]:
# ── Compute firing rate and recording duration from spike times ───────────────
import glob
dict_firing_rate   = {}
dict_rec_duration  = {}

for pkl_path in sorted(glob.glob(
        os.path.join(SPE1_PICKLE_ROOT, 'cluster_pickles', 'c*_cluster_df.pkl'))):
    cnum = int(os.path.basename(pkl_path).replace('_cluster_df.pkl', '').lstrip('c'))
    with open(pkl_path, 'rb') as f:
        cdf = pickle.load(f)
    t_ms = cdf['spk_times_ms'].values
    dur_s = (t_ms.max() - t_ms.min()) / 1000.0
    dict_firing_rate[cnum]  = len(t_ms) / dur_s          # Hz
    dict_rec_duration[cnum] = dur_s / 60.0                # minutes

# ── Build metadata DataFrame ──────────────────────────────────────────────────
df = build_metadata_df(
    all_results, cell_ids, target_names, target_labels, predictor_sets,
    DICT_CELL_TYPE, DICT_PATCH_TYPE, DICT_CORT_DEPTH, DICT_DARK_NEURONS, DICT_EAP_WAV,
    dict_firing_rate=dict_firing_rate,
    dict_rec_duration=dict_rec_duration,
)
print(df[['cell_id', 'cell_type', 'n_spikes', 'firing_rate_hz',
          'rec_duration_min', 'n_sig_waveform_only', 'mean_r2_wv']].to_string(index=False))

## 1. Per-cell metadata DataFrame

One row per cell: anatomy (type, depth, dark-neuron), recording properties (patch type,
EAP visible), and ridge regression summary stats (mean R², n significant targets,
per-target R²/significance/alpha).  Built by `build_metadata_df` from the per-cell
result pickles and the spe-1 config lookup tables.

In [ ]:
plot_cell_sig_heatmap(df, cell_ids, target_names, target_labels)

## 1. Cell × target significance heatmap

Shows which cells have FDR-significant Waveform-only models for each LFP target.
Cells sorted top-to-bottom by total number of significant targets.
Fraction significant per target column shown below x-axis.

In [ ]:
plot_r2_by_group(df, target_names, feat_labels)

## 2. CV R² distributions by target group

Boxplot + per-cell strip for 5-fold CV R² (Waveform only model),
split by target group (Pre abs / Pre-BL / Post abs / Post-BL / Δ).
Median annotated below each feature. Reference line at R²=0.

In [ ]:
plot_alpha_by_group(df, target_names, feat_labels)

## 2b. Regularisation strength by target group

Best Ridge alpha (log₁₀) selected by cross-validation per cell.
High alpha = model needed strong regularisation → weak or noisy signal in that target.
Significant targets (Pre/Post LFP Amp and Std) should show lower alpha than non-significant ones.

In [ ]:
plot_metadata_vs_mean_r2(df)

## 3. Does cell identity predict R²?

Tests whether intrinsic cell properties explain *how well* a cell's waveform predicts LFP.

- **Categorical variables** (cell type, patch type, dark-neuron, EAP visibility): Kruskal-Wallis test, box + strip plots.
- **Continuous variable** (cortical depth): Spearman correlation, scatter + trend line.

In [ ]:
plot_n_spikes_confound(df)

## 4. N spikes confound

`n_spikes = firing_rate × recording_duration` — it conflates two things:
- **Cell activity** (high-firing cells accumulate more spikes regardless of recording length)
- **Recording length** (longer recordings yield more spikes regardless of firing rate)

Three panels show each separately so you can tell which (if either) is the real driver.
A recording-length confound is methodological; a firing-rate confound is biologically meaningful
(active cells may have more predictable waveforms).

In [ ]:
plot_duration_rate_vs_metadata(df)

## 5. Are recording duration and firing rate confounded with cell metadata?

If duration or firing rate is significantly associated with cell type, patch type, etc.,
then any metadata effect on R² could be partially explained by recording differences
rather than cell biology. Significant panels (orange) flag potential confounds that
should be noted or controlled for in the R² × metadata interpretation.

In [ ]:
run_beta_metadata_analysis(beta_pop, df_r2, df, target_names, target_labels)

## 6. Beta direction × cell metadata

For each R²-significant LFP target (Wilcoxon median R²>0, FDR q<0.05):
test whether the *direction* of each waveform feature's beta weight can be explained by cell identity.

- Categorical metadata → Kruskal-Wallis (η² effect size; Δmedian for 2-group).
- Continuous metadata → Spearman ρ.
- FDR corrected (BH, q<0.05) within each target.

Always shows top-6 pairs by raw p regardless of FDR significance, so trends are visible
even at n=37. Scatter plots coloured by cell type to spot confounds.

In [ ]:
plot_predictor_set_comparison(df, predictor_sets)

## 7. Predictor set comparison

Does adding log-ISI improve prediction beyond waveform features alone?
Left: distribution of n-significant targets per cell (Waveform only).
Right: mean n-significant per predictor set with Wilcoxon paired tests vs Waveform only.